# 第 3 章 · RMSNorm 均方根归一化

> 📄 配套动画网页：`../03_rmsnorm.html`　（先用浏览器打开，把原理动画看一遍再来写代码）

**所属部分**：第二部分 · 模型

从这一章开始进入模型。第一个零件是归一化——它保证深层网络里激活值不会越跑越偏。

---


## 🎯 本章目标

- 理解 LayerNorm 与 RMSNorm 的区别
- 掌握 `torch.rsqrt`、`keepdim=True` 的广播规则
- 理解为什么需要 `keepdim`，写错会怎样
- 能从零实现 RMSNorm 并与 `nn.LayerNorm` 对照

---


## 🖼 官方配图

![RMSNorm 在整体架构中的位置：每个 Block 的注意力子层与前馈子层之前各有一个](../images/LLM-structure.jpg)

*RMSNorm 在整体架构中的位置：每个 Block 的注意力子层与前馈子层之前各有一个*

---


## 📖 背景：为什么需要归一化

深度网络里，每一层的输出都会被下一层放大或缩小。层数一多，数值就会指数级漂移——
要么爆炸成 `inf`，要么消失成 0。归一化把每层的数值**拉回同一量级**，训练才稳得住。

| | LayerNorm | RMSNorm |
|---|---|---|
| 减均值 | ✅ | ❌ |
| 除标准差 | ✅ | 除**均方根** |
| 偏置 β | ✅ | ❌ |
| 速度 | 慢 | 快 |

RMSNorm 的公式只有一行：

$$\text{RMS}(x)=\sqrt{\frac{1}{d}\sum_i x_i^2+\epsilon}, \qquad \text{RMSNorm}(x)=\gamma \odot \frac{x}{\text{RMS}(x)}$$

---



# 第一部分 · 基础函数演示

> 这一部分**可以直接运行**。先把本章要用到的 Python / PyTorch 函数和方法摸熟，后面自己写的时候才不会卡在 API 上。


### 3.1 幂与开方：`Tensor.pow`、`torch.sqrt`、`torch.rsqrt`

In [ ]:
import torch

x = torch.tensor([4.0, 9.0, 16.0])
print("x^2        :", x.pow(2))
print("sqrt(x)    :", torch.sqrt(x))
print("rsqrt(x)   :", torch.rsqrt(x))          # 1/sqrt(x)，比 1/torch.sqrt 更快更稳
print("验证        :", torch.allclose(torch.rsqrt(x), 1 / torch.sqrt(x)))

### 3.2 求均值与 `keepdim=True`

**这是本章最容易错的地方。** 看下面的形状变化。

In [ ]:
x = torch.randn(2, 5)          # 2 个样本，每个 5 维

m1 = x.mean(dim=-1)            # 形状 [2]
m2 = x.mean(dim=-1, keepdim=True)  # 形状 [2, 1]
print("x       :", x.shape)
print("mean    :", m1.shape, " keepdim:", m2.shape)

# 用 [2] 去减 [2,5] 会怎样？
try:
    print((x - m1).shape)
except RuntimeError as e:
    print("广播失败：", str(e).split("at")[0].strip())

print("keepdim 版本可以正确广播:", (x - m2).shape)

### 3.3 可学习参数：`nn.Parameter`

`nn.Parameter` 是会被 `optimizer` 更新的张量。RMSNorm 的 `γ` 就是它。

In [ ]:
from torch import nn

class Tiny(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))   # 初始化为全 1

t = Tiny(4)
print("weight        :", t.weight)
print("需要梯度吗     :", t.weight.requires_grad)
print("会被列出为参数 :", [n for n, _ in t.named_parameters()])

### 3.4 手动算一遍 RMSNorm

先不写类，用张量运算手推一遍，看清每一步的形状。

In [ ]:
x = torch.tensor([[2.0, -1.0, 3.0, -0.5]])
gamma = torch.tensor([1.2, 0.8, 1.0, 1.1])
eps = 1e-6

ms = x.pow(2).mean(dim=-1, keepdim=True)     # 均方
rms = torch.rsqrt(ms + eps)                  # 1/均方根
xn = x * rms                                 # 归一化
y = gamma * xn                               # 缩放

print("均方   :", ms.squeeze().item())
print("RMS    :", (1 / rms).squeeze().item())
print("归一化 :", xn.tolist())
print("输出   :", y.tolist())
print("归一化后的 RMS 应约等于 1:", xn.pow(2).mean().sqrt().item())

# 第二部分 · 动手实现（只给提示，留白给你）

> 下面每一节都**没有答案**。请根据文字提示 + 第一部分演示过的函数，自己从零写出来。
> 写不出来时，先回去看第一部分的演示，再对照网页里的动画想清楚数据形状。

模型部分的实现，只给提示。请自己写出来。

> 💡 每个留白代码块后面都跟了一个**测试块**。先写代码，再运行测试块。


In [ ]:
# ════════════════════ 自测工具 ════════════════════
# 每个留白代码块后面都跟一个测试块。先在上面写代码，再运行测试块。
# 没写之前运行也不会报错，只会提示「未实现」。
import torch
import torch.nn.functional as F

_PASS, _FAIL, _SKIP = [], [], []

def check(name, fn):
    """运行一个测试。捕获异常并打印结果，不会中断后面的测试。"""
    try:
        fn()
        print(f"✅ 通过   {name}")
        _PASS.append(name)
    except NameError as e:
        print(f"⏸  未实现 {name} —— {e}")
        _SKIP.append(name)
    except AssertionError as e:
        print(f"❌ 失败   {name} —— {e}")
        _FAIL.append(name)
    except Exception as e:
        print(f"❌ 出错   {name} —— {type(e).__name__}: {e}")
        _FAIL.append(name)

def close(a, b, tol=1e-4, msg=""):
    """断言两个张量 / 数字足够接近"""
    a, b = torch.as_tensor(a).float().reshape(-1), torch.as_tensor(b).float().reshape(-1)
    assert a.numel() == b.numel(), f"元素个数不一致 {a.numel()} vs {b.numel()} {msg}"
    assert torch.allclose(a, b, atol=tol), f"数值不接近（容差 {tol}）{msg}"

def same(a, b, msg=""):
    """断言两个张量完全相同"""
    a, b = torch.as_tensor(a), torch.as_tensor(b)
    assert a.shape == b.shape, f"形状不一致 {tuple(a.shape)} vs {tuple(b.shape)} {msg}"
    assert torch.equal(a, b), f"内容不相同 {msg}"

def summary():
    print()
    print(f"通过 {len(_PASS)} · 失败 {len(_FAIL)} · 未实现 {len(_SKIP)}")
    if _FAIL: print("需要修的：", "、".join(_FAIL))
    if _SKIP: print("还没写的：", "、".join(_SKIP))
    if _PASS and not _FAIL and not _SKIP:
        print("本章全部通过 🎉")

print("自测工具已就绪 —— 每写完一个留白，就运行它下面的测试块。")

### 3.5 实现 `RMSNorm` 类

**要求**：
- `__init__(self, dim, eps=1e-6)`：保存 `eps`，创建形状为 `dim` 的 `nn.Parameter`（初始全 1）
- `norm(x)`：返回 `x * rsqrt(mean(x²) + eps)`
- `forward(x)`：先算归一化，再乘 `weight`，**并保证输出 dtype 与输入一致**

**小坑**：MiniMind 里先 `.float()` 算归一化再 `.type_as(x)` 转回去——
因为 fp16 下平方容易溢出。

In [ ]:
# ======== 你的实现：RMSNorm ========
#
# 【要求】
#   - 继承 nn.Module
#   - weight = nn.Parameter(torch.ones(dim))
#   - norm(x) = x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + eps)
#   - forward(x) = (weight * norm(x.float())).type_as(x)
#
# 【可用到的函数 / 方法】
#   - `torch.rsqrt`
#   - `Tensor.mean(dim=-1, keepdim=True)`
#   - `Tensor.type_as(other)`
#   - `Tensor.float()`
#
# 提示：先想清楚输入输出的形状，再动手。
#
# ↓↓↓ 在这里写你的代码 ↓↓↓




In [ ]:
# ════════════ 测试：RMSNorm ════════════
# 先在上面把代码写出来，再运行这一格。
def _t():
    torch.manual_seed(0)
    m = RMSNorm(8)
    x = torch.randn(2, 5, 8)
    y = m(x)
    assert tuple(y.shape) == (2, 5, 8), f"输出形状应与输入一致，得到 {tuple(y.shape)}"
    rms = y.pow(2).mean(-1).sqrt()
    close(rms.mean().item(), 1.0, tol=1e-2, msg="weight 初始为 1 时，归一化后每个向量的 RMS 应约等于 1")
    close(m(x * 7.3), y, tol=1e-3, msg="RMSNorm 应具有尺度不变性：RMSNorm(k*x) == RMSNorm(x)")
    assert torch.isfinite(m(x * 1e4)).all(), "输入放大 1e4 倍后出现了 inf/nan，检查是否先用 float 计算"

check("RMSNorm 形状 / 数值 / 尺度不变性", _t)

### 3.6 验证：与 `nn.LayerNorm` 对比 + 尺度不变性

写完之后跑这两组验证：
1. 当 `weight` 全 1 且输入均值为 0 时，RMSNorm 与 LayerNorm 输出应该接近；
2. `RMSNorm(k * x) == RMSNorm(x)`（尺度不变性）。

In [ ]:
# ======== 你的实现：validate_rmsnorm ========
#
# 【要求】
#   - 构造一个随机输入 x 和你的 RMSNorm 实例
#   - 测试 1：比较 RMSNorm(x) 与 nn.LayerNorm(dim, elementwise_affine=False)(x) 的差异
#   - 测试 2：验证 RMSNorm(x) 与 RMSNorm(10 * x) 是否几乎相同
#   - 打印两种情况的误差
#
# 【可用到的函数 / 方法】
#   - `torch.allclose(a, b, atol=1e-3)`
#   - `nn.LayerNorm(dim, elementwise_affine=False)`
#
# 提示：先想清楚输入输出的形状，再动手。
#
# ↓↓↓ 在这里写你的代码 ↓↓↓




In [ ]:
summary()

# 第三部分 · 自测与思考

- [ ] 能解释为什么 `keepdim=True` 不能省
- [ ] 能说清 RMSNorm 比 LayerNorm 少了哪几步计算
- [ ] 尺度不变性验证通过了吗？
- [ ] 你的实现里 fp16 输入会溢出吗？加了 `.float()` 吗？

**思考题**：如果 `weight` 初始化成全 0 而不是全 1，训练一开始会发生什么？
